In [ ]:
!pip install -q -U transformers
!pip install -q -U accelerate
!pip install -q -U bitsandbytes
!pip install -q -U peft
!pip install -q -U datasets
!pip install -q -U evaluate
!pip install -q -U rouge_score
!pip install -q -U bert_score
!pip install -q -U nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

import torch

torch.backends.cuda.matmul.allow_tf32 = True

torch.set_default_dtype(torch.float16)

In [ ]:
import os
import nltk
import evaluate
import numpy as np

from datasets import Dataset

from transformers import (

    AutoTokenizer,

    AutoModelForCausalLM,

    BitsAndBytesConfig,

    TrainingArguments,

    DataCollatorForLanguageModeling,

    Trainer
)

from peft import (

    LoraConfig,

    get_peft_model,

    prepare_model_for_kbit_training
)

from bert_score import score


from nltk.translate.bleu_score import sentence_bleu

nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
base_path = "/content/drive/MyDrive/BBC News Summary/"

articles = []
summaries = []

article_dir = base_path + "News Articles/business"
summary_dir = base_path + "Summaries/business"

files = sorted(os.listdir(article_dir))

# SMALL SUBSET FOR T4
for f in files[:100]:

    with open(article_dir + "/" + f, "r", encoding="latin1") as file:

        article = file.read().replace("\n", " ").strip()

    with open(summary_dir + "/" + f, "r", encoding="latin1") as file:

        summary = file.read().replace("\n", " ").strip()

    articles.append(article)

    summaries.append(summary)


In [ ]:
texts = []

for article, summary in zip(articles, summaries):

    text = f"""
Summarize this article:

{article}

Summary:
{summary}
"""

    texts.append(text)

dataset = Dataset.from_dict({

    "text": texts
})

dataset = dataset.train_test_split(test_size=0.1)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 90
    })
    test: Dataset({
        features: ['text'],
        num_rows: 10
    })
})


In [ ]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# SAFE CONFIG FOR T4
bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=False
)

# TOKENIZER
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

# MODEL
model = AutoModelForCausalLM.from_pretrained(

    model_name,

    quantization_config=bnb_config,

    torch_dtype=torch.float16,

    device_map="auto"
)

# FIX PAD TOKEN
model.config.pad_token_id = tokenizer.pad_token_id

# DISABLE CACHE
model.config.use_cache = False

# PREPARE FOR QLORA
model = prepare_model_for_kbit_training(model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:

rouge = evaluate.load("rouge")

meteor = evaluate.load("meteor")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
baseline_predictions = []
baseline_references = []

print("\n========== BASELINE OUTPUT ==========\n")

for i in range(5):

    prompt = f"""
Summarize this article:

{articles[i]}

Summary:
"""

    inputs = tokenizer(

        prompt,

        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(

        **inputs,

        max_new_tokens=80
    )

    prediction = tokenizer.decode(

        outputs[0],

        skip_special_tokens=True
    )

    baseline_predictions.append(prediction)

    baseline_references.append(summaries[i])

    print(f"\n========== SAMPLE {i+1} ==========\n")

    print(prediction)


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



========== BASELINE OUTPUT ==========



[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



========== SAMPLE 1 ==========


Summarize this article:

Ad sales boost Time Warner profit  Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (Â£600m) for the three months to December, from $639m year-earlier.  The firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and higher advert sales. TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn. Its profits were buoyed by one-off gains which offset a profit dip at Warner Bros, and less users for AOL.  Time Warner said on Friday that it now owns 8% of search-engine Google. But its own internet business, AOL, had has mixed fortunes. It lost 464,000 subscribers in the fourth quarter profits were lower than in the preceding three quarters. However, the company said AOL's underlying profit before exceptional items rose 8% on the back of stronger internet advertising revenues. It hopes to increase subscribers by offering the online service free 

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



========== SAMPLE 2 ==========


Summarize this article:

Dollar gains on Greenspan speech  The dollar has hit its highest level against the euro in almost three months after the Federal Reserve head said the US trade deficit is set to stabilise.  And Alan Greenspan highlighted the US government's willingness to curb spending and rising household savings as factors which may help to reduce it. In late trading in New York, the dollar reached $1.2871 against the euro, from $1.2974 on Thursday. Market concerns about the deficit has hit the greenback in recent months. On Friday, Federal Reserve chairman Mr Greenspan's speech in London ahead of the meeting of G7 finance ministers sent the dollar higher after it had earlier tumbled on the back of worse-than-expected US jobs data. "I think the chairman's taking a much more sanguine view on the current account deficit than he's taken for some time," said Robert Sinche, head of currency strategy at Bank of America in New York. "He's taking a l

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



========== SAMPLE 3 ==========


Summarize this article:

Yukos unit buyer faces loan claim  The owners of embattled Russian oil giant Yukos are to ask the buyer of its former production unit to pay back a $900m (Â£479m) loan.  State-owned Rosneft bought the Yugansk unit for $9.3bn in a sale forced by Russia to part settle a $27.5bn tax claim against Yukos. Yukos' owner Menatep Group says it will ask Rosneft to repay a loan that Yugansk had secured on its assets. Rosneft already faces a similar $540m repayment demand from foreign banks. Legal experts said Rosneft's purchase of Yugansk would include such obligations. "The pledged assets are with Rosneft, so it will have to pay real money to the creditors to avoid seizure of Yugansk assets," said Moscow-based US lawyer Jamie Firestone, who is not connected to the case. Menatep Group's managing director Tim Osborne told the Reuters news agency: "If they default, we will fight them where the rule of law exists under the international arbi

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



========== SAMPLE 4 ==========


Summarize this article:

High fuel prices hit BA's profits  British Airways has blamed high fuel prices for a 40% drop in profits.  Reporting its results for the three months to 31 December 2004, the airline made a pre-tax profit of Â£75m ($141m) compared with Â£125m a year earlier. Rod Eddington, BA's chief executive, said the results were "respectable" in a third quarter when fuel costs rose by Â£106m or 47.3%. BA's profits were still better than market expectation of Â£59m, and it expects a rise in full-year revenues.  To help offset the increased price of aviation fuel, BA last year introduced a fuel surcharge for passengers.  In October, it increased this from Â£6 to Â£10 one-way for all long-haul flights, while the short-haul surcharge was raised from Â£2.50 to Â£4 a leg. Yet aviation analyst Mike Powell of Dresdner Kleinwort Wasserstein says BA's estimated annual surcharge revenues - Â£160m - will still be way short of its additional fuel costs 

In [ ]:
baseline_rouge = rouge.compute(

    predictions=baseline_predictions,

    references=baseline_references,

    use_stemmer=True
)

baseline_meteor = meteor.compute(

    predictions=baseline_predictions,

    references=baseline_references
)

P, R, F1 = score(

    baseline_predictions,

    baseline_references,

    lang="en",

    verbose=True
)

baseline_bertscore = F1.mean().item()

# BLEU
bleu1_scores = []
bleu4_scores = []

for i in range(len(baseline_predictions)):

    reference = [baseline_references[i].split()]

    candidate = baseline_predictions[i].split()

    bleu1 = sentence_bleu(

        reference,

        candidate,

        weights=(1, 0, 0, 0)
    )

    bleu4 = sentence_bleu(

        reference,

        candidate,

        weights=(0.25, 0.25, 0.25, 0.25)
    )

    bleu1_scores.append(bleu1)

    bleu4_scores.append(bleu4)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.96 seconds, 5.22 sentences/sec


In [ ]:
print("\n========== BASELINE METRICS ==========\n")

print("ROUGE-1 :", baseline_rouge["rouge1"])

print("ROUGE-2 :", baseline_rouge["rouge2"])

print("ROUGE-L :", baseline_rouge["rougeL"])

print("ROUGE-LSUM :", baseline_rouge["rougeLsum"])

print("\nBLEU-1 :", np.mean(bleu1_scores))

print("BLEU-4 :", np.mean(bleu4_scores))

print("\nMETEOR :", baseline_meteor["meteor"])

print("\nBERTScore :", baseline_bertscore)


========== BASELINE METRICS ==========

ROUGE-1 : 0.5693920909933965
ROUGE-2 : 0.5494472748033923
ROUGE-L : 0.35202297167644303
ROUGE-LSUM : 0.35455472381505226

BLEU-1 : 0.3677961479760467
BLEU-4 : 0.34321608900409833

METEOR : 0.6593325443286674

BERTScore : 0.8988186120986938


In [ ]:
lora_config = LoraConfig(

    r=8,

    lora_alpha=16,

    target_modules=[

        "q_proj",

        "k_proj",

        "v_proj",

        "o_proj"
    ],

    lora_dropout=0.05,

    bias="none",

    task_type="CAUSAL_LM"
)

# APPLY LORA
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 6,815,744 || all params: 7,248,547,840 || trainable%: 0.0940


In [ ]:
def tokenize_function(example):

    return tokenizer(

        example["text"],

        truncation=True,

        padding="max_length",

        max_length=512
    )

tokenized_dataset = dataset.map(tokenize_function)

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorForLanguageModeling(

    tokenizer=tokenizer,

    mlm=False
)


In [ ]:
training_args = TrainingArguments(

    output_dir="/content/drive/MyDrive/mistral_summary_model",

    num_train_epochs=3,

    per_device_train_batch_size=1,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_steps=10,

    save_steps=20,

    save_total_limit=2,

    fp16=False,

    bf16=False,

    report_to="none"
)

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_dataset["train"],

    data_collator=data_collator
)



In [ ]:
trainer.train()


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.442969
20,1.514844
30,1.358594
40,1.348437
50,1.296094
60,1.228906


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=69, training_loss=1.3421648550724639, metrics={'train_runtime': 638.4575, 'train_samples_per_second': 0.423, 'train_steps_per_second': 0.108, 'total_flos': 5903519160729600.0, 'train_loss': 1.3421648550724639, 'epoch': 3.0})

In [ ]:
save_path = "/content/drive/MyDrive/final_mistral_summary_model"

model.save_pretrained(save_path)

tokenizer.save_pretrained(save_path)

print("\nMODEL SAVED SUCCESSFULLY")


MODEL SAVED SUCCESSFULLY


In [ ]:
after_rouge = rouge.compute(

    predictions=after_predictions,

    references=after_references,

    use_stemmer=True
)

after_meteor = meteor.compute(

    predictions=after_predictions,

    references=after_references
)

P2, R2, F12 = score(

    after_predictions,

    after_references,

    lang="en",

    verbose=True
)

after_bertscore = F12.mean().item()

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.76 seconds, 6.55 sentences/sec


In [ ]:
print("\n========== AFTER TRAINING METRICS ==========\n")

print("ROUGE-1 :", after_rouge["rouge1"])

print("ROUGE-2 :", after_rouge["rouge2"])

print("ROUGE-L :", after_rouge["rougeL"])

print("ROUGE-LSUM :", after_rouge["rougeLsum"])

print("\nMETEOR :", after_meteor["meteor"])

print("\nBERTScore :", after_bertscore)


========== AFTER TRAINING METRICS ==========

ROUGE-1 : 0.5979900972666535
ROUGE-2 : 0.5760785271456876
ROUGE-L : 0.370939891049427
ROUGE-LSUM : 0.37163797132865917

METEOR : 0.6669555054428604

BERTScore : 0.8939316868782043


In [ ]:
print("\n========== IMPROVEMENT ==========\n")

print({

    "ROUGE-1 Improvement":
        after_rouge["rouge1"] - baseline_rouge["rouge1"],

    "ROUGE-2 Improvement":
        after_rouge["rouge2"] - baseline_rouge["rouge2"],

    "ROUGE-L Improvement":
        after_rouge["rougeL"] - baseline_rouge["rougeL"],

    "ROUGE-LSUM Improvement":
        after_rouge["rougeLsum"] - baseline_rouge["rougeLsum"],

    "METEOR Improvement":
        after_meteor["meteor"] - baseline_meteor["meteor"],

    "BERTScore Improvement":
        after_bertscore - baseline_bertscore
})


========== IMPROVEMENT ==========

{'ROUGE-1 Improvement': np.float64(0.028598006273257037), 'ROUGE-2 Improvement': np.float64(0.026631252342295353), 'ROUGE-L Improvement': np.float64(0.018916919372983954), 'ROUGE-LSUM Improvement': np.float64(0.017083247513606903), 'METEOR Improvement': np.float64(0.007622961114193028), 'BERTScore Improvement': -0.004886925220489502}
